# Análisis Exploratorio de Datos (EDA) - Mercado Global de Videojuegos
**Autor:** [Álvaro Domingo Cordón / Enlace a tu LinkedIn]  
**Repositorio:** [Enlace a tu GitHub]  
**Área:** Inteligencia de Negocio & Data Analytics  

---

### 🎯 Objetivo del Análisis
El propósito de este notebook es transformar datos brutos de la industria de los videojuegos (histórico de VGChartz) en insights estratégicos accionables. Respondemos de forma sistemática a las siguientes preguntas clave de negocio:
1. **Evolución Temporal:** ¿Cómo ha crecido la industria en volumen de lanzamientos frente a ingresos globales?
2. **Categorías Dominantes:** ¿Qué plataformas y géneros concentran el monopolio comercial?
3. **Asimetría Geográfica:** ¿Existen patrones culturales de consumo diferenciados por regiones?
4. **Impacto de la Crítica:** ¿Qué tan fuerte es la correlación real entre una buena calificación y el éxito comercial masivo?
5. **Estacionalidad (Ventanas Estratégicas):** ¿Cuál es la mejor época del año para lanzar un videojuego al mercado?

# 2. Librerías.

In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns",None)

df = pd.read_csv("../Data/vgchartz-2024.csv")


# 3. carga del dataset.

In [46]:
print("=== INSPECCIÓN DE COLUMNAS ORIGINALES ===")
columnas_originales = df.columns.tolist()
print(columnas_originales)

# 2. Diccionario de mapeo dinámico (Detecta variaciones comunes sin inventar datos)
mapeo_columnas = {}
for col in columnas_originales:
    col_low = col.lower().strip()
    
    if col_low in ['year', 'year_of_release', 'release_year']:
        mapeo_columnas[col] = 'year'
    elif col_low in ['total_sales', 'global_sales', 'sales', 'total_sales_millions']:
        mapeo_columnas[col] = 'total_sales'
    elif col_low in ['genre', 'genero']:
        mapeo_columnas[col] = 'genre'
    elif col_low in ['console', 'platform', 'plataforma']:
        mapeo_columnas[col] = 'console'
    elif col_low in ['critic_score', 'score', 'critic']:
        mapeo_columnas[col] = 'critic_score'
    elif col_low in ['release_date', 'date', 'fecha']:
        mapeo_columnas[col] = 'release_date'
    elif col_low in ['na_sales', 'na_sales_millions']:
        mapeo_columnas[col] = 'na_sales'
    elif col_low in ['jp_sales', 'jp_sales_millions']:
        mapeo_columnas[col] = 'jp_sales'
    elif col_low in ['pal_sales', 'eu_sales', 'europe_sales', 'pal_sales_millions']:
        mapeo_columnas[col] = 'pal_sales'
    elif col_low in ['other_sales', 'others_sales']:
        mapeo_columnas[col] = 'other_sales'

# Apply the mapping
df = df.rename(columns=mapeo_columnas)

# 3. Respaldo por si 'year' no existía pero sí hay una fecha completa
if 'year' not in df.columns and 'release_date' in df.columns:
    print("\n[INFO] 'year' no encontrado directamente. Extrayendo desde 'release_date'...")
    df['year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Convertir tipos clave de forma segura
if 'year' in df.columns:
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

print("\n=== COLUMNAS TRADUCIDAS Y LISTAS PARA EL EDA ===")
print(df.columns.tolist())


=== INSPECCIÓN DE COLUMNAS ORIGINALES ===
['img', 'title', 'console', 'genre', 'publisher', 'developer', 'critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales', 'release_date', 'last_update', 'year']

=== COLUMNAS TRADUCIDAS Y LISTAS PARA EL EDA ===
['img', 'title', 'console', 'genre', 'publisher', 'developer', 'critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales', 'release_date', 'last_update', 'year']


In [47]:
df.shape
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   img           64016 non-null  str    
 1   title         64016 non-null  str    
 2   console       64016 non-null  str    
 3   genre         64016 non-null  str    
 4   publisher     64016 non-null  str    
 5   developer     63999 non-null  str    
 6   critic_score  6678 non-null   float64
 7   total_sales   18922 non-null  float64
 8   na_sales      12637 non-null  float64
 9   jp_sales      6726 non-null   float64
 10  pal_sales     12824 non-null  float64
 11  other_sales   15128 non-null  float64
 12  release_date  56965 non-null  str    
 13  last_update   17879 non-null  str    
 14  year          56965 non-null  Int64  
dtypes: Int64(1), float64(6), str(8)
memory usage: 7.4 MB


### Con esta primera carga de datos podemos hacernos una idea de las filas y las columas de las que dispone nuestro dataset, además nos damos cuenta que la columna release_date es un str.Por eso creamos una nueva columna llama year con el nuevo valor.

# 4. Limpieza de datos.

## Duplicados


In [50]:
# Calcular duplicados totales
duplicados_totales = df.duplicated().sum()
print(f" Filas completamente idénticas detectadas: {duplicados_totales}")

 Filas completamente idénticas detectadas: 0


In [54]:
if duplicados_totales > 0:
    # Eliminar duplicados
    df_cleaned = df_raw.drop_duplicates(keep='first')
    print(f" han eliminado {duplicados_totales} filas redundantes.")
    print(f"Dimensiones actuales del dataset: {df_cleaned.shape[0]} filas.")
else:
    df_cleaned = df.copy()
    print("No se encontraron filas duplicadas en el dataset.")

No se encontraron filas duplicadas en el dataset.
